# Урок 08 - Шаблон проектування багатокористувацьких агентів


## Налаштування


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Чому мультиагентні системи?

Задачі з реального світу, як-от планування поїздки, потребують різних видів експертизи — логістики, місцевих знань, бюджету тощо. Один агент, який намагається впоратися з усім, швидко стає незручним.

Мультиагентні системи вирішують це за допомогою **спеціалізації**: кожен агент зосереджується на одній області експертизи, що дає результати вищої якості, ніж у загального фахівця. Вони також покращують **масштабованість** — можна додавати нових агентів (наприклад, фахівця з авіаперельотів, критика ресторанів) без переписування існуючого робочого процесу. Агенти працюють разом через структурований конвеєр, передаючи контекст від одного до іншого.


## Створення спеціалізованих агентів


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Побудова послідовного робочого процесу

`WorkflowBuilder` дозволяє з'єднувати агентів у орієнтований граф. Тут ми створюємо простий двокроковий конвеєр: **TravelPlanner** складає маршрут, а потім **TravelConcierge** переглядає та покращує його.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавання більше агентів до робочого процесу

Однією з найбільших переваг патерну з кількома агентами є те, як легко його розширити. Нижче ми додаємо агента **BudgetReviewer**, який перевіряє план відповідно до бюджету мандрівника, позначає пункти, що можуть призвести до перевищення ліміту витрат, та пропонує альтернативи для економії коштів. Тепер робочий процес послідовно запускає трьох агентів:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Підсумок

У цьому уроці ви дізналися, як:

1. **Створювати спеціалізованих агентів** — кожен із зосередженою роллю (планування, консьєрж, перегляд бюджету).
2. **Підключати агентів до послідовного робочого процесу** за допомогою `WorkflowBuilder` і `add_edge`.
3. **Потоково виводити** результат з конвеєра багатьох агентів, відстежуючи, який агент говорить.
4. **Розширювати робочий процес**, додаючи нових агентів у ланцюжок без зміни існуючих.

Дизайн багатьох агентів дозволяє кожному агенту бути простим, водночас створюючи багатші, більш ретельно перевірені результати, яких не може досягти жоден агент поодинці.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Положення про відповідальність**:
Цей документ було перекладено за допомогою сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоч ми й прагнемо до точності, будь ласка, враховуйте, що автоматичний переклад може містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звертатися до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
